# 01 — Data Exploration

**DiaFoot.AI v2 | Phase 1, Commit 2**

This notebook explores the downloaded DFU datasets:
- FUSeg (1,210 images from 889 patients)
- AZH wound segmentation (1,010 images)
- DFUC 2022 (4,000 images — if license obtained)

We analyze: image counts, resolution distributions, file sizes, format breakdown,
mask statistics, and CleanVision quality audit results.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

DATA_ROOT = Path("../data/raw/dfu")
METADATA_DIR = Path("../data/metadata")

print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Exists: {DATA_ROOT.exists()}")

## 1. Dataset File Counts

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

def count_images(directory: Path) -> dict:
    """Count images recursively, separating images from masks."""
    if not directory.exists():
        return {"total": 0, "images": 0, "masks": 0}

    all_files = [f for f in directory.rglob("*") if f.suffix.lower() in IMAGE_EXTS]
    masks = [f for f in all_files if "label" in str(f).lower() or "mask" in str(f).lower()]
    images = [f for f in all_files if f not in masks]
    return {"total": len(all_files), "images": len(images), "masks": len(masks)}

datasets = {"fuseg": DATA_ROOT / "fuseg", "azh": DATA_ROOT / "azh", "dfuc2022": DATA_ROOT / "dfuc2022"}

for name, path in datasets.items():
    counts = count_images(path)
    print(f"{name:10s} | images: {counts['images']:5d} | masks: {counts['masks']:5d} | total: {counts['total']:5d}")

## 2. Image Resolution Distribution

In [ ]:
def get_image_sizes(directory: Path, max_samples: int = 500) -> list[tuple[int, int]]:
    """Get (width, height) for images in directory."""
    if not directory.exists():
        return []
    files = [f for f in directory.rglob("*") if f.suffix.lower() in IMAGE_EXTS
             and "label" not in str(f).lower() and "mask" not in str(f).lower()]
    sizes = []
    for f in files[:max_samples]:
        try:
            with Image.open(f) as img:
                sizes.append(img.size)
        except Exception:
            pass
    return sizes

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, path) in zip(axes, datasets.items()):
    sizes = get_image_sizes(path)
    if sizes:
        widths, heights = zip(*sizes)
        ax.scatter(widths, heights, alpha=0.3, s=10)
        ax.set_xlabel("Width (px)")
        ax.set_ylabel("Height (px)")
        ax.set_title(f"{name} (n={len(sizes)})")
    else:
        ax.set_title(f"{name} (no data)")
plt.tight_layout()
plt.savefig(str(METADATA_DIR / "resolution_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Sample Images & Masks

In [ ]:
def show_samples(dataset_path: Path, dataset_name: str, n: int = 4):
    """Show sample image-mask pairs."""
    img_dirs = list(dataset_path.rglob("images"))
    if not img_dirs:
        print(f"No 'images' subdirectory found in {dataset_path}")
        return

    img_dir = img_dirs[0]
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])[:n]

    if not imgs:
        print(f"No images found in {img_dir}")
        return

    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for i, img_path in enumerate(imgs):
        # Show image
        img = Image.open(img_path)
        axes[0, i].imshow(img)
        axes[0, i].set_title(img_path.name[:20], fontsize=8)
        axes[0, i].axis("off")

        # Try to find corresponding mask
        mask_path = None
        for mask_dir_name in ["labels", "masks"]:
            candidate = img_path.parent.parent / mask_dir_name / img_path.name
            if candidate.exists():
                mask_path = candidate
                break
            # Try with different extension
            for ext in [".png", ".jpg"]:
                candidate = img_path.parent.parent / mask_dir_name / (img_path.stem + ext)
                if candidate.exists():
                    mask_path = candidate
                    break

        if mask_path and mask_path.exists():
            mask = Image.open(mask_path)
            axes[1, i].imshow(mask, cmap="gray")
            axes[1, i].set_title("mask", fontsize=8)
        else:
            axes[1, i].text(0.5, 0.5, "No mask", ha="center", va="center")
        axes[1, i].axis("off")

    fig.suptitle(f"{dataset_name} — Sample Images & Masks", fontsize=14)
    plt.tight_layout()
    plt.show()

for name, path in datasets.items():
    if path.exists() and any(path.rglob("*")):
        show_samples(path, name)

## 4. CleanVision Quality Audit Results

In [ ]:
# Load the combined quality report
report_path = METADATA_DIR / "quality_report.json"
if report_path.exists():
    with open(report_path) as f:
        quality_report = json.load(f)

    for ds_name, ds_report in quality_report.get("datasets", {}).items():
        print(f"\n{'=' * 50}")
        print(f"Dataset: {ds_name}")
        print(f"Total images: {ds_report['total_images']}")
        print("Issues:")
        for issue, count in ds_report.get("summary", {}).items():
            pct = count / ds_report['total_images'] * 100 if ds_report['total_images'] > 0 else 0
            print(f"  {issue}: {count} ({pct:.1f}%)")
else:
    print("Quality report not found. Run: python scripts/run_cleaning.py")

## 5. Mask Statistics (wound coverage)

In [ ]:
def analyze_masks(dataset_path: Path, max_samples: int = 200) -> dict:
    """Analyze mask coverage (% of wound pixels per image)."""
    mask_dirs = list(dataset_path.rglob("labels")) + list(dataset_path.rglob("masks"))
    if not mask_dirs:
        return {}

    coverages = []
    for mask_dir in mask_dirs:
        masks = sorted([f for f in mask_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])[:max_samples]
        for m in masks:
            try:
                mask = np.array(Image.open(m).convert("L"))
                wound_pct = (mask > 0).sum() / mask.size * 100
                coverages.append(wound_pct)
            except Exception:
                pass

    if not coverages:
        return {}
    return {
        "count": len(coverages),
        "mean_coverage_pct": round(np.mean(coverages), 2),
        "median_coverage_pct": round(np.median(coverages), 2),
        "min_coverage_pct": round(min(coverages), 2),
        "max_coverage_pct": round(max(coverages), 2),
        "coverages": coverages,
    }

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, path) in zip(axes, datasets.items()):
    stats = analyze_masks(path)
    if stats and stats.get("coverages"):
        ax.hist(stats["coverages"], bins=30, edgecolor="black", alpha=0.7)
        ax.axvline(stats["mean_coverage_pct"], color="red", linestyle="--", label=f"mean={stats['mean_coverage_pct']:.1f}%")
        ax.set_xlabel("Wound coverage (%)")
        ax.set_ylabel("Count")
        ax.set_title(f"{name} (n={stats['count']})")
        ax.legend()
    else:
        ax.set_title(f"{name} (no masks)")
plt.tight_layout()
plt.savefig(str(METADATA_DIR / "mask_coverage_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary

| Dataset | Images | Masks | Source | Status |
|---------|--------|-------|--------|--------|
| FUSeg | 1,210 | 1,010+ | GitHub (public) | ✓ Downloaded |
| AZH | 1,010 | 1,010 | GitHub (public) | ✓ Downloaded |
| DFUC 2022 | 4,000 | 2,000 | License required | ⏳ Pending |

**Next steps (Commit 3-7):**
1. Collect healthy foot images (negative class)
2. Collect non-DFU foot conditions (hard negatives)
3. Cleanlab label quality audit
4. ITA skin tone analysis
5. Preprocessing pipeline + stratified splits